# Module 3: ReAct — Interactive Planning (WebShop-Style Tasks)

**Stanford CS329A: Self-Improving AI Agents · Personal Study Notebook**  
Reading: [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) (Yao et al., 2022 / ICLR 2023)

### Research question
**Does revising a plan based on observations help an agent search, inspect, and correctly configure an item before making a decision?**

### Notebook structure

- **Part 0 — Setup & Dataset EDA:** Learn the shopping-goal format, product attributes, and environment state.
- **Part 1 — Greedy baseline:** Search once, pick the first item, and default to its first listed options.
- **Part 2 — Act-only:** Choose actions from observations without an explicit plan.
- **Part 3 — ReAct:** Add one short plan before each action.
- **Part 4 — Evaluation:** Compare task success, mistakes, steps, and model costs.
- **Part 5 — Failure analysis:** Investigate which action caused a bad purchase or step-limit failure.

**Where the code lives:** `helpers.py` contains the simulator, prompting, agent loop, and metrics. `data/` stores readable **JSON product and goal fixtures**. The notebook is a walkthrough and experiment, not a wall of function definitions.

> **Scope:** This experiment uses a **small synthetic WebShop-style simulator**, not the official WebShop evaluation environment. Its rewards are educational, **not** official benchmark scores. A separate optional section introduces a real WebShop product-data preview; do not conflate its products with the synthetic task results.

## Part 0 — Setup & EDA

WebShop-style tasks are **stateful**: an action changes what the agent can see and do next. To complete a request, it may need to **search → open a product → read its choices → choose the correct options → purchase**.

In Colab, run the next cell to clone or update the GitHub checkout and enter the WebShop directory. In local Jupyter, start the kernel in `03_react/` with the adjacent helper and fixture files. The experiment uses simulated purchases.

### Thought → Action → Observation

A thought uses the current goal and prior observations to guide the next action. An action queries or changes the environment, and the observation reports the result. The next decision should use that result, especially after an invalid action or an incomplete product match. Act-only controls for access to the same tools and observations while omitting the explicit thought step.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/dariuscparker/Self-Improving-Agents-Lab.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Self-Improving-Agents-Lab")
WEBSHOP_PATH = Path("modules/03_learning_from_feedback_with_tools/03_react")

webshop_dir = Path.cwd()
if "google.colab" in sys.modules:
    if (REPO_DIR / ".git").is_dir():
        current_branch = subprocess.check_output(["git", "-C", str(REPO_DIR), "branch", "--show-current"], text=True).strip()
        if current_branch != REPO_BRANCH:
            raise RuntimeError(f"Switch the checkout at {REPO_DIR} to {REPO_BRANCH} before rerunning setup.")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", REPO_URL, REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    webshop_dir = REPO_DIR / WEBSHOP_PATH

required_files = ("helpers.py", "data/mini_products.json", "data/mini_goals.json", "requirements.txt")
missing_files = [name for name in required_files if not (webshop_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Missing {', '.join(missing_files)} in {webshop_dir}. "
                            "Check the GitHub branch or start Jupyter in the WebShop directory.")

os.chdir(webshop_dir)
if str(webshop_dir) not in sys.path:
    sys.path.insert(0, str(webshop_dir))
print(f"WebShop directory: {Path.cwd()}")

In [ ]:
%pip -q install -r requirements.txt

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import helpers as hh
pd.set_option('display.max_colwidth', 160)

### Shared Experiment Parameters

Use the **same six goals**, product list, action budget, model, and decoding settings in all experiments. API calls are off initially so you can study the deterministic environment and baseline without a hosted model or GPU.

In [ ]:
SEED = 42  # fixtures and simulator are deterministic; retained for future extensions
RUN_LIVE_LLM = False
PROVIDER = 'groq'  # or 'openai'; requires matching API key if enabled
LOAD_REAL_WEBSHOP_SAMPLE = False  # exploratory preview only; not used for scoring
CFG = {'model': 'llama-3.3-70b-versatile' if PROVIDER == 'groq' else 'gpt-4o-mini',
       'temperature': 0.0, 'max_output_tokens': 170, 'max_steps': 10,
       'observation_chars': 1200, 'results_limit': 5}
print({'seed': SEED, 'live_llm': RUN_LIVE_LLM, **CFG})

### Understand the shopping dataset

The original WebShop benchmark couples product records with natural-language instructions and a website-like action environment. To make action/reward behavior easy to inspect, this notebook uses two tiny **synthetic JSON files**:

| File | Main fields | Purpose |
|---|---|---|
| `data/mini_products.json` | `id`, `title`, `description`, `price`, `options` | The visible catalog |
| `data/mini_goals.json` | `id`, `instruction`, `target_id`, `required` | Agent's instruction and evaluator-only target |

**Crucial separation:** the agent receives the instruction and rendered product pages, **not** the expected `target_id` or `required` choices. The simulator keeps those targets internally to calculate reward when the episode ends.

**Task to solve:** Select the correct item **and every requested option**. The mini-simulator gives binary reward 1 for a perfect match, 0 otherwise. The original WebShop benchmark has more complex rewards and a much larger catalog.

In [ ]:
products, goals = hh.load_fixtures('data')
print(f'Catalog: {len(products)} products | Evaluation: {len(goals)} fixed tasks')
display(hh.catalog_table(products))
display(hh.goal_table(goals))

### Study one product and one instruction

Notice that `options` is a dictionary of attribute names to possible values. A correct **product ID alone** does not guarantee success: the agent must click the exact requested **size, color, scent, or capacity** as well.

The goal's target and required options are intentionally **not shown to the agent**. The next display is for **your study notes** only.

In [ ]:
print('PRODUCT SCHEMA EXAMPLE:')
print(json.dumps(products[0], indent=2))
print('\nGOAL INSTRUCTION:', goals[0]['instruction'])
print('Ground truth exists in mini_goals.json; used only by the evaluator.')

### Actions, observations, and state

Legal actions: `search[query]`, `click[visible product ID]`, `click[visible option value]`, `click[Back to Search]`, `click[Buy Now]`.

The environment gives **observations** after actions (which product page is open, available sizes, selected options). The agent cannot click arbitrary invisible product IDs or assume a selection worked.

Walk through **one example by hand**. This is a guided demonstration, **not** a model-generated result or a benchmark score.

In [ ]:
store = hh.MiniWebShop(products)
print(store.reset(goals[0]))
for action in ['search[sensitive skin citrus deodorant]', 'click[D100]',
               'click[bright citrus]', 'click[3 oz]', 'click[Buy Now]']:
    observation, reward, done, info = store.step(action)
    print(f'\nACTION: {action}\nOBSERVATION: {observation[:210]}')
print('\nFinal reward (hand-authored demonstration):', reward)
assert done and reward == 1.0

### Why this is an agent problem

The model must repeatedly **choose its next action**, *observe* the result, then decide what to do. The correct path is not just a single answer: a wrong variant or premature `Buy Now` ends the episode with reward 0.

In `helpers.py`, `MiniWebShop` handles the state machine. An `Action` is executed by the simulator; `Thought` does not itself mutate state.

In [ ]:
wrong = hh.MiniWebShop(products)
wrong.reset(goals[0])
for action in ['search[citrus deodorant]', 'click[D100]',
               'click[calming lavender]', 'click[3 oz]', 'click[Buy Now]']:
    _, wrong_reward, _, _ = wrong.step(action)
print('Correct product + wrong scent => reward:', wrong_reward)
assert wrong_reward == 0.0

## Part 1 — Greedy Baseline: Choose Too Quickly

**Policy:** Convert the instruction into one broad search query, click the **first result**, select the **first listed value** for every option, and buy.

This is intentionally weak. It uses the task instruction but **never reads the evaluator's target ID or required attributes**. It helps demonstrate why a multi-step policy may be necessary.

**Prediction:** Easy defaults may succeed, but tasks with misleading first results or nondefault variants should fail.

In [ ]:
greedy_df = hh.evaluate(goals, products, cfg=CFG)
display(greedy_df[['id', 'instruction', 'success', 'steps', 'purchased_id', 'selection']])

## Part 2 — Act-Only: Actions Based on Observations

**Policy:** An LLM decides the next `search[...]` or `click[...]` using the task and current observed page. It is **not prompted** to write an explicit planning note.

The model sees only visible pages and must inspect variants before selecting them. Its actions and any invalid clicks are logged by the helper.

**Controlled comparison:** Both Act-only and ReAct use the *same* goals, rendered observations, action budget, and model settings.

In [ ]:
client = hh.build_client(PROVIDER) if RUN_LIVE_LLM else None
act_df = (hh.model_results(goals, products, ['Act'], client, CFG)
          if RUN_LIVE_LLM else pd.DataFrame())
print('Act-only scored episodes:', len(act_df))

## Part 3 — ReAct: Plan → Act → Observe → Update

**Policy:** Before issuing an action, the model may add a **short `Thought:` line** stating what it needs to inspect next. The simulator executes the `Action:` line and supplies a fresh `Observation:`.

*Reading connection:* ReAct §4 and the WebShop trajectory examples in Appendix D.3. Observe whether the agent **changes its search or product choice** after an unsatisfactory observation.

**Hypothesis:** Explicit planning helps avoid premature purchase, missing product variants, and repeated invalid actions. It may also increase token cost.

In [ ]:
react_df = (hh.model_results(goals, products, ['ReAct'], client, CFG)
            if RUN_LIVE_LLM else pd.DataFrame())
print('ReAct scored episodes:', len(react_df))

## Part 4 — Evaluate on Identical Goals

**Primary outcome:** Exact task success (`reward=1` only for the desired product **and** all specified options).

**Efficiency and errors:** Steps, invalid actions, provider-reported total tokens (if available), and API latency. The deterministic greedy baseline does **not** consume model tokens; its CPU wall time is **not directly comparable** with hosted model latency.

With `RUN_LIVE_LLM=False`, only the **real, executed greedy baseline** appears; no invented Act/ReAct scores are filled in. This is a **mini-simulator result**, not an official WebShop benchmark score.

In [ ]:
runs = pd.concat([greedy_df, act_df, react_df], ignore_index=True)
summary = hh.summarize(runs)
display(summary.round(3))
print('API errors (excluded from aggregate):', (runs['status'] == 'api_error').sum())

In [ ]:
ax = summary.set_index('method')['success_rate'].plot.bar(rot=0, figsize=(8, 4))
ax.set_ylim(0, 1)
ax.set_ylabel('Exact success rate')
ax.set_title('Synthetic WebShop-style simulator — NOT official benchmark scores')
plt.tight_layout(); plt.show()

### Interpret the results, not only the scores

An improvement can result from *better search queries*, *better choices among plausible items*, *checking variants*, or *recovering from misleading observations*. More actions are not inherently better; inspect whether the extra actions improved success.

Since tasks and products are synthetic and few in number, treat this as a **mechanism demonstration**, not evidence of general WebShop performance.

## Part 5 — Failure Analysis and Trajectory Inspection

Review the actual `Action` → `Observation` sequence and identify the first meaningful deviation:

1. Was the first search query too broad or too narrow?
2. Did the agent click a plausible but incorrect item?
3. Did it select **every required option**?
4. Did it attempt an action unavailable on the current page?
5. Did it terminate too early or run out of steps?
6. Did ReAct's brief plan help recover from an observation?

Only after the episode ends, you can inspect `mini_goals.json` to see the correct target and option labels.

In [ ]:
display(runs[runs['success'] == 0][['id', 'method', 'purchased_id',
                                    'selection', 'steps', 'invalid_actions']])
row = runs[runs['status'] != 'api_error'].iloc[0]
print('GOAL:', row['instruction'], '\nMETHOD:', row['method'])
print(json.dumps(row['trace'], indent=2))

In [ ]:
folder = hh.save_results(runs)
print('Saved scores.csv and trajectories.jsonl to', folder)

### Optional: inspect authentic WebShop product data

The actual WebShop environment is substantially more complex. This optional download examines a **third-party mirror of the 1,000-product preview**, which may change or become unavailable. This data is **not used by the simulator** and does not provide official tasks or scores.

If you want to use the official benchmark later, read the original repository's setup instructions: [princeton-nlp/WebShop](https://github.com/princeton-nlp/WebShop). Its environment and reward system must be run separately.

In [ ]:
if LOAD_REAL_WEBSHOP_SAMPLE:
    preview = hh.sample_real_webshop_products()
    print('Preview type:', type(preview).__name__, 'items:', len(preview))
    if isinstance(preview, list) and preview:
        print('First item keys:', list(preview[0]) if isinstance(preview[0], dict) else type(preview[0]))
        display(pd.DataFrame(preview[:2]))
else:
    print('Optional real-product EDA is off; synthetic experiment unaffected.')

### Ablations for a follow-up session

- **Action budget:** Try `max_steps=5` versus 10 versus 15. What failures can extra steps resolve?
- **Observation size:** Truncate pages to 250 characters; check whether options disappear from view.
- **Harder catalog:** Add near-duplicates with conflicting sizes or descriptions to `mini_products.json`.
- **Query quality:** Change the number of search results and evaluate whether agent search improves.
- **Plan ablation:** Compare Act-only and ReAct while holding the model and task goals fixed; log token cost as well as action count.

Make changes to the small JSON fixtures for dataset experiments. Edit the simulator or reward only when explicitly studying **environment design**; note that such a change creates a different benchmark.

## Conclusions & Study Notes

**Technique learned:** ReAct is an *interleaved decision loop*. In a stateful environment, actions are consequential: a good answer or good-looking product is not enough if the agent executes the wrong clicks.

**What is demonstrated:** An observable shopping task schema; a deterministic simulator; a naive baseline; Act-only and ReAct model policies; legal actions, invalid actions, reward, trajectory analysis, and efficiency metrics.

**What isn't reproduced:** Real WebShop task distribution, public evaluation goals, product search index, website dynamics, original model prompts, or historical benchmark scores.

**Reading reflections:**
- Find a trace where a new observation made an earlier plan obsolete.
- Find a case where the correct product was chosen but the configuration was wrong.
- Did explicit planning improve decisions enough to justify any extra cost?


Sources: [ReAct](https://arxiv.org/abs/2210.03629) · [WebShop](https://github.com/princeton-nlp/WebShop) · [ReAct author's implementation](https://github.com/ysymyth/ReAct).